# B20 - Forward Diffusion on EMNIST-Letters (Offset Cosine schedule, 1200 steps)
a. Corrupt images gradually over **1200 steps** using an **offset cosine** noise schedule.  b. Show noisy images at **t = 200 and t = 1200**.

Forward process (closed form): $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$, with $\epsilon\sim\mathcal{N}(0,I)$

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
import tensorflow_datasets as tfds
ds = tfds.load('emnist/letters', split='train', as_supervised=True)
imgs, labels = [], []
for img, lab in ds.take(6):
    imgs.append(tf.transpose(img, [1, 0, 2])[:, :, 0].numpy())   # EMNIST images are stored transposed
    labels.append(chr(64 + int(lab)))                            # 1 -> 'A'
x0 = np.array(imgs).astype("float32") / 127.5 - 1.0           # scale to [-1, 1]
print("Letters:", labels)

In [ ]:
# Offset cosine schedule, T = 1200 (signal rate goes from 0.95 down to 0.02)
def offset_cosine_schedule(T, min_signal_rate=0.02, max_signal_rate=0.95):
    t = np.arange(T + 1) / T
    start_angle = np.arccos(max_signal_rate)
    end_angle = np.arccos(min_signal_rate)
    angles = start_angle + t * (end_angle - start_angle)
    alpha_bar = np.cos(angles) ** 2          # signal_rate^2
    return alpha_bar

alpha_bar = offset_cosine_schedule(1200)

In [ ]:
class ForwardDiffusion:
    def __init__(self, alpha_bar):
        self.alpha_bar = tf.constant(alpha_bar, dtype=tf.float32)    # index t = 0..T

    def q_sample(self, x0, t):
        ab = tf.gather(self.alpha_bar, t)
        eps = tf.random.normal(tf.shape(x0))
        return tf.sqrt(ab) * x0 + tf.sqrt(1.0 - ab) * eps

T = 1200
diffusion = ForwardDiffusion(alpha_bar)

plt.plot(alpha_bar); plt.xlabel('t'); plt.ylabel('alpha_bar_t'); plt.title('Noise schedule'); plt.show()

In [ ]:
# Gradual corruption of one image
steps = np.linspace(0, T, 9).astype(int)
plt.figure(figsize=(14, 2))
for i, t in enumerate(steps):
    plt.subplot(1, len(steps), i + 1)
    plt.imshow(diffusion.q_sample(x0[0], int(t)), cmap='gray'); plt.title(f't={t}'); plt.axis('off')
plt.suptitle('Gradual corruption'); plt.show()

In [ ]:
# Required steps: t = [200, 1200]
show = [200, 1200]
n = len(x0)
plt.figure(figsize=(2 * n, 2 * (len(show) + 1)))
for i in range(n):
    plt.subplot(len(show) + 1, n, i + 1); plt.imshow(x0[i], cmap='gray'); plt.axis('off')
    if i == 0: plt.title('original', loc='left')
    for r, t in enumerate(show):
        plt.subplot(len(show) + 1, n, (r + 1) * n + i + 1)
        plt.imshow(diffusion.q_sample(x0[i], t), cmap='gray'); plt.axis('off')
        if i == 0: plt.title(f't={t}', loc='left')
plt.show()

for t in show:
    print(f"t={t}: alpha_bar={alpha_bar[t]:.4f}  signal weight={np.sqrt(alpha_bar[t]):.4f}  noise weight={np.sqrt(1-alpha_bar[t]):.4f}")